# 10 · Roads from OpenStreetMap with osmnx

[osmnx](https://osmnx.readthedocs.io/) downloads the OpenStreetMap road network of any place on Earth.
roadstyle draws it. Together they turn a place name into a styled, interactive, offline map in a few lines,
with no data file of your own.

Needs osmnx:  `pip install osmnx`

In [1]:
import osmnx as ox
import pandas as pd
import roadstyle as rs

# Any place name that OpenStreetMap's geocoder knows: a district, city, county, …
G = ox.graph_from_place("Södermalm, Stockholm", network_type="drive", simplify=False)
# Simplify ourselves, but never merge across a bridge or tunnel boundary (see below).
G = ox.simplify_graph(G, edge_attrs_differ=["bridge", "tunnel"])
print(f"{G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges ({type(G).__name__})")

739 nodes, 1,470 edges (MultiDiGraph)


### From an osmnx graph to road edges
`graph_to_gdfs` gives the edges as a GeoDataFrame, already with the `highway` column roadstyle styles by.

Two details of osmnx data:

- **The graph is directed.** A two-way street is two edges, one per direction, and a one-way street is one.
  That is exactly what roadstyle expects: it draws the two directions side by side and puts arrows on
  one-way streets. Keep the graph directed; don't convert it with `ox.convert.to_undirected`.
- **Simplification can merge a tunnel with a bridge.** By default osmnx joins consecutive OSM ways into one
  edge, so Söderledstunneln and Centralbron become a single 1.5 km edge tagged both `tunnel` and `bridge`,
  and the whole tunnel would be drawn as a bridge. That is why the download above uses `simplify=False`
  and then `ox.simplify_graph(G, edge_attrs_differ=["bridge", "tunnel"])`: roads still merge, but never
  across the start or end of a bridge or tunnel.
- **Some cells hold lists.** A merged edge can carry several values, such as
  `name = ['Götgatan', 'Ringvägen']`, and osmnx indexes edges by `(u, v, key)`. roadstyle (0.9 and
  later) takes this as it is: it keeps the first value of each list and turns the index into columns.

In [2]:
edges = ox.graph_to_gdfs(G, nodes=False)
edges[["name", "highway", "oneway", "maxspeed", "length"]].head()

name   highway  oneway  maxspeed  \
u      v          key                                                  
119024 34586358   0    Söder Mälarstrand  tertiary   False        40   
       21651762   0       Pålsundsbacken  tertiary   False  [50, 40]   
       35123521   0    Söder Mälarstrand  tertiary   False        40   
119039 1886783086 0    Söder Mälarstrand  tertiary    True        40   
       1221343891 0    Söder Mälarstrand  tertiary   False        40   

                           length  
u      v          key              
119024 34586358   0    295.754525  
       21651762   0    109.921187  
       35123521   0     76.768431  
119039 1886783086 0    130.317380  
       1221343891 0    230.657147

In [3]:
m = rs.render_edges(edges, compress=True)
m.save("output/osmnx_sodermalm.html")
m

### Style by any OSM tag
osmnx keeps the OSM tags as columns, so anything from `data-driven styling` (notebook 2) works on them.
Here, the speed limit: `maxspeed` is text in OSM (`"30"`), so make it a number first.

In [4]:
edges["speed_kmh"] = pd.to_numeric(edges["maxspeed"], errors="coerce")
rs.render_edges(edges, color_by="speed_kmh", cmap="plasma", legend=True, compress=True)

### Other places and networks
The same two lines work for any area, only the download changes:

| You want… | osmnx call |
|---|---|
| a named place | `ox.graph_from_place("Tartu, Estonia", simplify=False)` |
| around a point | `ox.graph_from_point((59.315, 18.07), dist=1000, simplify=False)` |
| a bounding box | `ox.graph_from_bbox((west, south, east, north), simplify=False)` |
| a polygon you have | `ox.graph_from_polygon(poly, simplify=False)` |

each followed by `ox.simplify_graph(G, edge_attrs_differ=["bridge", "tunnel"])`, and `network_type` picks the roads: `"drive"`, `"walk"`, `"bike"` or `"all"`. Large areas download slowly
the first time; osmnx caches the result, so the next run is quick.

In short:
```python
G = ox.simplify_graph(ox.graph_from_place(place, simplify=False), edge_attrs_differ=["bridge", "tunnel"])
rs.render_edges(ox.graph_to_gdfs(G, nodes=False))
```